# SingleTaskVariationalGP

This notebook demonstrates `robotorchan.models.SingleTaskVariationalGP`, an approximate GP for datasets where exact GP training becomes expensive.

The example uses inducing points and a minibatch training loop with `VariationalELBO`.

## 1. When to use this model

Use a variational GP when the cubic scaling of an exact GP becomes a practical bottleneck. The approximation is controlled primarily by the number and placement of inducing points.

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, TensorDataset

from robotorchan.models import SingleTaskVariationalGP

torch.manual_seed(0)
dtype = torch.double

## 2. Synthetic training data

In [ ]:
def objective(x: torch.Tensor) -> torch.Tensor:
    return torch.sin(2.0 * torch.pi * x) + 0.25 * torch.cos(6.0 * torch.pi * x)

n_train = 400
train_X = torch.rand(n_train, 1, dtype=dtype)
train_Y = objective(train_X) + 0.08 * torch.randn(n_train, 1, dtype=dtype)

train_X.shape, train_Y.shape

## 3. Model construction

In [ ]:
model = SingleTaskVariationalGP(
    train_X=train_X,
    train_Y=train_Y,
    inducing_points=48,
)
model

## 4. robotorchan common API

The wrapper retains the caller-supplied tensors and provides an ELBO factory. `train_Yvar` is not part of the upstream variational constructor, so `raw_train_Yvar` is `None`.

In [ ]:
print("raw_train_X:", model.raw_train_X.shape)
print("raw_train_Y:", model.raw_train_Y.shape)
print("raw_train_Yvar:", model.raw_train_Yvar)
print("raw_data_names:", model.raw_data_names)
print("supports_mll:", model.supports_mll)

mll = model.make_mll(num_data=n_train)
type(mll).__name__

## 5. Minibatch variational training

`num_data` must represent the total dataset size even though each ELBO evaluation sees only one minibatch. The trainable approximate GP is available as `model.model`, while the likelihood remains `model.likelihood`.

In [ ]:
loader = DataLoader(
    TensorDataset(train_X, train_Y),
    batch_size=64,
    shuffle=True,
)

model.train()
model.likelihood.train()
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
mll = model.make_mll(num_data=n_train)

loss_history = []
for epoch in range(60):
    epoch_loss = 0.0
    for batch_X, batch_Y in loader:
        optimizer.zero_grad()
        output = model.model(batch_X)
        loss = -mll(output, batch_Y.squeeze(-1))
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    loss_history.append(epoch_loss / len(loader))

print(f"initial loss: {loss_history[0]:.3f}")
print(f"final loss:   {loss_history[-1]:.3f}")

In [ ]:
plt.figure(figsize=(7, 3))
plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("Negative ELBO")
plt.title("Variational training history")
plt.show()

## 6. Posterior prediction

In [ ]:
model.eval()
model.likelihood.eval()

test_X = torch.linspace(0.0, 1.0, 250, dtype=dtype).unsqueeze(-1)
with torch.no_grad():
    posterior = model.posterior(test_X)
    mean = posterior.mean.squeeze(-1)
    std = posterior.variance.sqrt().squeeze(-1)

lower = mean - 1.96 * std
upper = mean + 1.96 * std

In [ ]:
plt.figure(figsize=(9, 5))
plt.scatter(train_X.squeeze(-1), train_Y.squeeze(-1), s=8, alpha=0.25, label="observations")
plt.plot(test_X.squeeze(-1), objective(test_X).squeeze(-1), linestyle="--", label="true function")
plt.plot(test_X.squeeze(-1), mean, label="posterior mean")
plt.fill_between(test_X.squeeze(-1), lower, upper, alpha=0.2, label="95% interval")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.title("SingleTaskVariationalGP posterior")
plt.show()

## 7. Exact GP versus variational GP

`SingleTaskGP` is usually the first choice for small datasets because exact inference is simple and statistically efficient. `SingleTaskVariationalGP` becomes attractive as the dataset grows and exact covariance factorization becomes costly. The inducing-point count controls the speed / fidelity trade-off.